In [ ]:

# RTO MODEL – FINAL OOF BOOSTED


import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import KFold
import joblib
import warnings
warnings.filterwarnings("ignore")


# 1. LOAD DATA


FILE = "DATA_RTO8_TRAIN_TEST_SPLIT.xlsx"

train = pd.read_excel(FILE, sheet_name="Train")
test  = pd.read_excel(FILE, sheet_name="Test")

train.columns = train.columns.str.strip()
test.columns  = test.columns.str.strip()


# 2. TARGET


valid_status = [12, 13, 14, 15]
train = train[train["StatusId"].isin(valid_status)].copy()
test  = test[test["StatusId"].isin(valid_status)].copy()

train["RTO"] = train["StatusId"].isin([13,14,15]).astype(int)
test["RTO"]  = test["StatusId"].isin([13,14,15]).astype(int)


# 3. BASIC CLEAN


for df in [train, test]:
    df["PickedUpOn"] = pd.to_datetime(df["PickedUpOn"], errors="coerce")
    df.dropna(subset=["PickedUpOn"], inplace=True)

    df["MobileNumber"] = df["MobileNumber"].astype(str)
    df["DestinationPincode"] = df["DestinationPincode"].astype(str)
    df["Courier"] = df["Courier"].astype(str)

    df["TotalOrderValue"] = df["TotalOrderValue"].clip(lower=1)
    df["Quantity"] = df["Quantity"].clip(lower=1)


# =========================================================
# MOBILE NORMALIZATION (CRITICAL FOR LOOKUP)
# =========================================================

def normalize_mobile(x):
    if pd.isna(x):
        return "UNKNOWN"
    x = str(x).strip()
    x = x.replace("+91", "")
    x = x.replace(" ", "")
    x = x.replace("-", "")
    x = x.replace(".0", "")
    return x

train["MobileNumber"] = train["MobileNumber"].apply(normalize_mobile)
test["MobileNumber"]  = test["MobileNumber"].apply(normalize_mobile)

# 4. USER HISTORY (LEAKAGE SAFE)


train = train.sort_values("PickedUpOn")

train["user_total_orders"] = train.groupby("MobileNumber").cumcount()
train["user_last_rto"] = train.groupby("MobileNumber")["RTO"].shift(1).fillna(0)
train["user_rto_count"] = train.groupby("MobileNumber")["user_last_rto"].cumsum()

train["user_rto_rate"] = np.where(
    train["user_total_orders"] > 0,
    train["user_rto_count"] / train["user_total_orders"],
    0
)

user_store = (
    train.groupby("MobileNumber")
    .agg(
        user_total_orders=("user_total_orders","max"),
        user_rto_rate=("user_rto_rate","max"),
        user_last_rto=("user_last_rto","last")
    )
)

# Ensure index is string and clean
user_store.index = user_store.index.astype(str).str.strip()

test = test.merge(user_store, on="MobileNumber", how="left").fillna(0)


# 5. PRODUCT CATEGORY


train["ProductCategory"] = train["ProductName"].fillna("UNK").apply(lambda x: str(x).split(" ")[0])
test["ProductCategory"]  = test["ProductName"].fillna("UNK").apply(lambda x: str(x).split(" ")[0])


# 6.  OOF TARGET ENCODING (PINCODE + CATEGORY)


global_mean = train["RTO"].mean()
alpha = 30

def oof_target_encode(train_df, test_df, col, target):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof = pd.Series(index=train_df.index, dtype=float)

    for tr_idx, val_idx in kf.split(train_df):
        tr, val = train_df.iloc[tr_idx], train_df.iloc[val_idx]

        stats = tr.groupby(col)[target].agg(["sum","count"])
        enc = (stats["sum"] + alpha * global_mean) / (stats["count"] + alpha)

        oof.iloc[val_idx] = val[col].map(enc)

    oof.fillna(global_mean, inplace=True)

    full_stats = train_df.groupby(col)[target].agg(["sum","count"])
    full_enc = (full_stats["sum"] + alpha * global_mean) / (full_stats["count"] + alpha)

    test_encoded = test_df[col].map(full_enc).fillna(global_mean)

    return oof, test_encoded


train["pin_rto"], test["pin_rto"] = oof_target_encode(train, test, "DestinationPincode", "RTO")
train["cat_rto"], test["cat_rto"] = oof_target_encode(train, test, "ProductCategory", "RTO")


# 7. ADDRESS SIGNAL


train["addr_len"] = train["CompleteAddress"].fillna("").str.len()
test["addr_len"]  = test["CompleteAddress"].fillna("").str.len()

train["addr_digit"] = train["CompleteAddress"].fillna("").str.count(r"\d")
test["addr_digit"]  = test["CompleteAddress"].fillna("").str.count(r"\d")


# 8. ORDER FEATURES


for df in [train, test]:
    df["is_cod"] = (df["PaymentMode"] == 1).astype(int)
    df["log_value"] = np.log1p(df["TotalOrderValue"])


# 9. BOOST FEATURES (SAFE RANK BOOST)


train["user_vs_global"] = train["user_rto_rate"] - global_mean
test["user_vs_global"]  = test["user_rto_rate"] - global_mean

train["pin_vs_global"] = train["pin_rto"] - global_mean
test["pin_vs_global"]  = test["pin_rto"] - global_mean

train["risk_logit_pin"] = np.log((train["pin_rto"]+1e-5)/(1-train["pin_rto"]+1e-5))
test["risk_logit_pin"]  = np.log((test["pin_rto"]+1e-5)/(1-test["pin_rto"]+1e-5))

train["risk_logit_user"] = np.log((train["user_rto_rate"]+1e-5)/(1-train["user_rto_rate"]+1e-5))
test["risk_logit_user"]  = np.log((test["user_rto_rate"]+1e-5)/(1-test["user_rto_rate"]+1e-5))

train["cod_x_user"] = train["is_cod"] * train["user_rto_rate"]
test["cod_x_user"]  = test["is_cod"] * test["user_rto_rate"]

train["cod_x_pin"] = train["is_cod"] * train["pin_rto"]
test["cod_x_pin"]  = test["is_cod"] * test["pin_rto"]


# 10. FINAL FEATURES


features = [
    "is_cod",
    "log_value",
    "Quantity",
    "Courier",
    "DestinationPincode",

    "user_total_orders",
    "user_rto_rate",
    "user_last_rto",

    "pin_rto",
    "cat_rto",

    "addr_len",
    "addr_digit",

    "user_vs_global",
    "pin_vs_global",
    "risk_logit_pin",
    "risk_logit_user",

    "cod_x_user",
    "cod_x_pin"
]

cat_features = ["Courier", "DestinationPincode"]


# 11. MODEL


model = CatBoostClassifier(
    iterations=5500,
    depth=9,
    learning_rate=0.02,
    l2_leaf_reg=12,
    bagging_temperature=1.1,
    random_strength=1.2,
    border_count=254,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=200
)

train_pool = Pool(train[features], train["RTO"], cat_features=cat_features)
test_pool  = Pool(test[features], test["RTO"], cat_features=cat_features)

model.fit(train_pool, eval_set=test_pool)


# 12. METRICS


probs = model.predict_proba(test_pool)[:,1]

roc = roc_auc_score(test["RTO"], probs)
pr  = average_precision_score(test["RTO"], probs)
brier = brier_score_loss(test["RTO"], probs)

print("\n FINAL ROC-AUC:", roc)
print("PR-AUC:", pr)
print("Brier:", brier)


# 13. SAVE


joblib.dump(model, "rto_model_final_oof.joblib")
joblib.dump(features, "features_final_oof.joblib")
# =========================================================
# 14. SAVE FEATURE STORES FOR PRODUCTION PREDICTION
# =========================================================

# ---- USER HISTORY STORE ----
# (Already computed above as user_store)
joblib.dump(user_store, "user_store.joblib")

# ---- PINCODE STORE (FULL ENCODING FROM TRAIN ONLY) ----
pin_full_stats = train.groupby("DestinationPincode")["RTO"].agg(["sum","count"])
pin_full_enc = (pin_full_stats["sum"] + alpha * global_mean) / (pin_full_stats["count"] + alpha)
joblib.dump(pin_full_enc, "pin_store.joblib")

# ---- CATEGORY STORE (FULL ENCODING FROM TRAIN ONLY) ----
cat_full_stats = train.groupby("ProductCategory")["RTO"].agg(["sum","count"])
cat_full_enc = (cat_full_stats["sum"] + alpha * global_mean) / (cat_full_stats["count"] + alpha)
joblib.dump(cat_full_enc, "cat_store.joblib")

# ---- GLOBAL MEAN ----
joblib.dump(global_mean, "global_mean.joblib")


print("\n✅ FEATURE STORES SAVED")

print("\n✅ MODEL SAVED")


0:	test: 0.8908310	best: 0.8908310 (0)	total: 517ms	remaining: 47m 21s
200:	test: 0.9022600	best: 0.9022600 (200)	total: 52.9s	remaining: 23m 15s
400:	test: 0.9022156	best: 0.9024830 (265)	total: 1m 42s	remaining: 21m 44s
600:	test: 0.9017448	best: 0.9024830 (265)	total: 2m 31s	remaining: 20m 35s
800:	test: 0.9016378	best: 0.9024830 (265)	total: 3m 23s	remaining: 19m 54s
1000:	test: 0.9012174	best: 0.9024830 (265)	total: 4m 29s	remaining: 20m 10s
1200:	test: 0.9003477	best: 0.9024830 (265)	total: 5m 20s	remaining: 19m 8s
1400:	test: 0.8998613	best: 0.9024830 (265)	total: 6m 13s	remaining: 18m 14s
1600:	test: 0.8994161	best: 0.9024830 (265)	total: 7m 7s	remaining: 17m 20s
1800:	test: 0.8992389	best: 0.9024830 (265)	total: 8m 3s	remaining: 16m 33s
2000:	test: 0.8990928	best: 0.9024830 (265)	total: 8m 58s	remaining: 15m 41s
2200:	test: 0.8988475	best: 0.9024830 (265)	total: 9m 53s	remaining: 14m 48s
2400:	test: 0.8987487	best: 0.9024830 (265)	total: 10m 51s	remaining: 14m
2600:	test: 0.89

In [6]:

# RTO PRODUCTION PREDICTION


import joblib
import pandas as pd
import numpy as np


# 1. LOAD ARTIFACTS


model        = joblib.load("rto_model_final_oof.joblib")
FEATURES     = joblib.load("features_final_oof.joblib")
user_store   = joblib.load("user_store.joblib")
pin_store    = joblib.load("pin_store.joblib")
cat_store    = joblib.load("cat_store.joblib")
GLOBAL_MEAN  = joblib.load("global_mean.joblib")

courier_perf = pd.read_csv("Courier_Percentage_Delivered.csv")
courier_perf = courier_perf.sort_values("PercentageDelivered", ascending=False)


# NORMALIZATION
def normalize_mobile(m):
    return str(m).replace("+91", "").replace(".0", "").strip()


# 3. SAFE USER LOOKUP (NO FAKE HISTORY)


def lookup_user(mobile):
    mobile = normalize_mobile(mobile)

    if mobile in user_store.index:
        row = user_store.loc[mobile]
        return {
            "user_total_orders": int(row.user_total_orders),
            "user_rto_rate": float(row.user_rto_rate),
            "user_last_rto": int(row.user_last_rto),
            "source": "CUSTOMER_HISTORY"
        }

    return {
        "user_total_orders": 0,
        "user_rto_rate": 0.0,
        "user_last_rto": 0,
        "source": "NEW_USER"
    }

def lookup_pincode(pin):
    return pin_store.get(str(pin), GLOBAL_MEAN)


# 4. FEATURE BUILDER (MATCHES TRAINING EXACTLY)


def build_features(
    mobile_no,
    address,
    destination_pincode,
    order_value,
    quantity,
    payment_mode,
):
    user = lookup_user(mobile_no)
    pin_rto = lookup_pincode(destination_pincode)

    is_cod = int(payment_mode == 1)
    log_value = np.log1p(order_value)

    addr = str(address)
    addr_len = len(addr)
    addr_digit = sum(c.isdigit() for c in addr)

    row = {
        "is_cod": is_cod,
        "log_value": log_value,
        "Quantity": quantity,
        "Courier": "UNKNOWN",
        "DestinationPincode": str(destination_pincode),

        "user_total_orders": user["user_total_orders"],
        "user_rto_rate": user["user_rto_rate"],
        "user_last_rto": user["user_last_rto"],

        "pin_rto": pin_rto,
        "cat_rto": GLOBAL_MEAN,

        "addr_len": addr_len,
        "addr_digit": addr_digit,

        "user_vs_global": user["user_rto_rate"] - GLOBAL_MEAN,
        "pin_vs_global": pin_rto - GLOBAL_MEAN,

        "risk_logit_pin": np.log((pin_rto + 1e-6)/(1 - pin_rto + 1e-6)),
        "risk_logit_user": (
            np.log((user["user_rto_rate"] + 1e-6)/(1 - user["user_rto_rate"] + 1e-6))
            if user["user_total_orders"] > 0 else 0
        ),

        "cod_x_user": is_cod * user["user_rto_rate"],
        "cod_x_pin": is_cod * pin_rto
    }

    df = pd.DataFrame([row])
    df = df.reindex(columns=FEATURES, fill_value=0)

    return df, user, pin_rto


# 5. RISK BAND LOGIC


def risk_band(prob):
    pct = prob * 100
    if pct < 30:
        return "LOW"
    elif pct < 50:
        return "MEDIUM"
    elif pct < 70:
        return "HIGH"
    return "VERY_HIGH"


# 6. FIXED DECISION SOURCE (NO MORE PINCODE BIAS)


def decision_source(user, pin_rate, prob):
    """
    Decide dominant risk source using RELATIVE strength.
    """

    # If low risk → no dominant factor
    if prob < 0.30:
        return "LOW_RISK_NO_DOMINANT_FACTOR"

    user_rate = user["user_rto_rate"]

    # Strong customer history
    if user["source"] == "CUSTOMER_HISTORY" and user_rate > pin_rate + 0.05:
        return "CUSTOMER_HISTORY"

    # Strong pincode
    if pin_rate > user_rate + 0.05:
        return "PINCODE_HISTORY"

    return "MIXED_RISK (USER + PINCODE)"


# 7. COURIER RECOMMENDATION


def recommend_top_couriers(n=5):
    return courier_perf.head(n)[
        ["Courier", "PercentageDelivered"]
    ].to_dict("records")


# 8. MAIN PREDICTION FUNCTION


def predict_rto(
    mobile_no,
    address,
    destination_pincode,
    order_value,
    quantity,
    payment_mode,
):
    X, user, pin_rto = build_features(
        mobile_no,
        address,
        destination_pincode,
        order_value,
        quantity,
        payment_mode,
    )

    prob = model.predict_proba(X)[0,1]
    band = risk_band(prob)

    src = decision_source(user, pin_rto, prob)

    action_map = {
        "LOW": "ALLOW_SHIPMENT",
        "MEDIUM": "CALL_CONFIRMATION",
        "HIGH": "OTP_OR_PARTIAL_PREPAID",
        "VERY_HIGH": "PREPAID_ONLY"
    }

    return {
        "rto_probability": round(prob,4),
        "rto_percentage": f"{round(prob*100,2)}%",
        "risk_band": band,
        "decision_source": src,
        "recommended_action": action_map[band],
        "recommended_couriers": recommend_top_couriers()
    }


# 9. EXAMPLE


if __name__ == "__main__":
    result = predict_rto(
        mobile_no="9999177550",
        address="E-204, laboni  crossing republik",
        destination_pincode="201016",
        order_value=300,
        quantity=1,
        payment_mode=1,
    )

    print(result)


{'rto_probability': np.float64(0.3213), 'rto_percentage': '32.13%', 'risk_band': 'MEDIUM', 'decision_source': 'PINCODE_HISTORY', 'recommended_action': 'CALL_CONFIRMATION', 'recommended_couriers': [{'Courier': 'VelocityExpress', 'PercentageDelivered': 91.82}, {'Courier': 'Blitz', 'PercentageDelivered': 84.21}, {'Courier': 'Delhivery', 'PercentageDelivered': 74.473636}, {'Courier': 'Amazon', 'PercentageDelivered': 70.785}, {'Courier': 'BlueDart', 'PercentageDelivered': 70.625714}]}
